# 💰 Regression with an Insurance Dataset

## 🎯 Objective

This notebook aims to analyze a dataset on insurance premiums with the goal of building a regression model to predict premium amounts. Insurance premiums, which depend on factors such as demographics, health indicators, lifestyle choices, and claims history, are critical for risk assessment and policy pricing. This study seeks to understand the relationships between these factors and premium amounts, providing insights to improve risk prediction and optimize insurance models.

"Why seek insurance when you can win big"
![](https://static01.nyt.com/images/2023/12/07/multimedia/07squid-game-highs-lows2-zvwf/07squid-game-highs-lows2-zvwf-superJumbo.jpg)
Source: Squid Games | Netflix

## 1. 📝 Introduction

Insurance premiums are a critical aspect of the insurance industry, directly impacting both individuals and companies. These premiums, which vary based on numerous factors such as demographics, health status, lifestyle, and previous claims, determine the cost of insurance policies. Understanding the factors that influence premium amounts can lead to more accurate pricing models and better risk management strategies.

In this project, we will leverage a dataset containing various features related to personal and health information to predict the target variable, Premium Amount. This problem will be approached as a regression task, where we aim to predict the continuous premium value based on these influencing factors.

Problem Statement: The goal is to predict the Premium Amount, a continuous variable, based on the input features provided in the dataset. This will involve using machine learning models to understand the relationships between the input variables and the premium amounts, ultimately minimizing the Root Mean Squared Logarithmic Error (RMSLE) to improve prediction accuracy.

In [ ]:
import pandas
import numpy
import os

import catboost
import lightgbm
import plotly.express
import plotly.graph_objects
import sklearn.preprocessing
import sklearn.ensemble
import sklearn.linear_model
import xgboost

DATA_DIR = "/kaggle/input/playground-series-s4e12"

## 2. 📊 Data Overview

In [ ]:
TRAINING_DATAFRAME = pandas.read_csv(os.path.join(DATA_DIR, "train.csv"))
TESTING_DATAFRAME  = pandas.read_csv(os.path.join(DATA_DIR, "test.csv"))

TRAINING_DATAFRAME.columns

The dataset includes the following features:

🧍 Demographic Information: `Age`, `Gender`, `Annual Income`, `Marital Status`, `Number of Dependents`, `Education Level`, `Occupation`, `Location`, `Property Type`.

💼 Financial and Risk Factors: `Credit Score`, `Previous Claims`, `Insurance Duration`.

💪 Health and Lifestyle Information: `Health Score`, `Smoking Status`, `Exercise Frequency`.

📑 Policy and Feedback Information: `Policy Type`, `Policy Start Date`, `Customer Feedback`.

Our target variable is `Premium Amount`, which we will predict based on these features.

In [ ]:
TRAINING_DATAFRAME.head()

In [ ]:
TRAINING_DATAFRAME.describe()

In [ ]:
TRAINING_DATAFRAME.describe(include='object')

## 3. 🚀 Project Workflow

### 3.1 🧹 Data Preprocessing: Clean and preprocess the data, handle missing values, encode categorical features, and scale numerical features as necessary.

In [ ]:
TRAINING_DATAFRAME.info()

In [ ]:
TRAINING_DATAFRAME.isnull().sum()

In [ ]:
pipeline = [
    # ("Column", function, "Column Name Append")
]

def run_pipeline(pipeline, dataframe, drop_updated_columns = True, verbose = True):
    columns_to_clean = set()
    for column_name, operation, column_name_append in pipeline:
        if verbose:
            print(f"Applying operation: {operation.__name__} to {column_name}")
        dataframe[f"{column_name}_{column_name_append}"] = operation(dataframe[column_name])
        columns_to_clean.add(column_name)

    if drop_updated_columns:
        if verbose:
            print(f"Dropping original columns: {list(columns_to_clean)}")
        dataframe = dataframe.drop(columns=list(columns_to_clean))

    return dataframe

Let us fix `Age` first. But before deciding what to do, let's see the data distribution of `Age`.

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Age"].value_counts().reset_index().sort_values(by = "Age"),
    x = "Age",
    y = "count",
    labels = {'index': 'Category', 'Category': 'Frequency'},
    title = "Distribution of Age",
    template = "plotly_dark").show(renderer = 'iframe')

An interesting problem involves addressing 18,000 missing age values in a dataset where the data is uniformly distributed. Three potential strategies to resolve this issue have been identified:

- Replacing the missing age values with the median age.
- Randomly sampling age values from the uniform distribution.
- Imputing missing age values using predictions from a model.
  
The focus will initially be on exploring the first strategy.

In [ ]:
def __impute_column_with_median(column):
    assert isinstance(column, pandas.Series)

    median = column.median(skipna = True)
    return column.fillna(median)

def __impute_column_with_random_sample_from_distribution(column):
    assert isinstance(column, pandas.Series)
    observed_values = column.dropna()
    assert not observed_values.empty
    return column.apply(lambda x: numpy.random.choice(observed_values) if pandas.isna(x) else x)

pipeline.append(("Age", __impute_column_with_median, "MEDIAN"))
pipeline.append(("Age", __impute_column_with_random_sample_from_distribution, "RANDOM"))

Let's understand how to approach missing values of `Annual Income` by analyzing the trend.

In [ ]:
plotly.express.histogram(
    TRAINING_DATAFRAME,
    x="Annual Income",
    nbins=50, 
    labels={'Annual_Salary': 'Annual Salary', 'count': 'Frequency'},
    title="Histogram of Annual Salaries",
    template="plotly_dark"
).show(renderer = "iframe")

There is a right skew to this data. Instead of using the `Median` to replace the missing values, it might be better to use `Q1` quartile. 

In [ ]:
def __impute_column_with_q1(column):
    assert isinstance(column, pandas.Series)

    q1 = column.quantile(0.25)
    return column.fillna(q1)

pipeline.append(("Annual Income", __impute_column_with_q1, "Q1"))

Let's now analyze `Marital Status`.

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Marital Status"].value_counts().reset_index(),
    x = "Marital Status", 
    y = "count", 
    labels={'index': 'Marital Status', 'Marital Status': 'Frequency'},
    title="Distribution of Marital Status",
    template="plotly_dark"
).show(renderer = "iframe")

And again, another uniform distribution. Let's replace the missing value with `Unknown` token.

In [ ]:
def __impute_column_with_mode(column):
    assert isinstance(column, pandas.Series)
    mode_value = column.mode().iloc[0]
    return column.fillna(mode_value)

def __impute_column_with_value(column, value = "UNKNOWN"):
    assert isinstance(column, pandas.Series)
    return column.fillna(value)

pipeline.append(("Marital Status", __impute_column_with_mode, "MODE"))
pipeline.append(("Marital Status", __impute_column_with_value, "W_VALUE"))
pipeline.append(("Marital Status", __impute_column_with_random_sample_from_distribution, "RANDOM"))

Let's hope the next one isn't another one of uniform distribution.

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Number of Dependents"].value_counts().reset_index(),
    x = "Number of Dependents", 
    y = "count", 
    labels={'index': 'Number of Dependents', 'Number of Dependents': 'Frequency'},
    title="Distribution of Number of Dependents",
    template="plotly_dark"
).show(renderer = "iframe")

Oh well... 🤦

In [ ]:
pipeline.append(("Number of Dependents", __impute_column_with_median, "MEDIAN"))
pipeline.append(("Number of Dependents", __impute_column_with_random_sample_from_distribution, "RANDOM"))
pipeline.append(("Number of Dependents", __impute_column_with_value, "W_VALUE"))

Time to fix `Occupation`.

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Occupation"].value_counts().reset_index(),
    x = "Occupation", 
    y = "count", 
    labels={'index': 'Occupation', 'Occupation': 'Frequency'},
    title="Distribution of Occupation",
    template="plotly_dark"
).show(renderer = "iframe")

In [ ]:
pipeline.append(("Occupation", __impute_column_with_mode, "MODE"))
pipeline.append(("Occupation", __impute_column_with_value, "W_VALUE"))
pipeline.append(("Occupation", __impute_column_with_random_sample_from_distribution, "RANDOM"))

Analyzing `Health Score`

In [ ]:
plotly.express.histogram(
    TRAINING_DATAFRAME,
    x="Health Score",
    nbins=50, 
    labels={'Health_Score': 'Health Score', 'count': 'Frequency'},
    title="Histogram of Health Score",
    template="plotly_dark"
).show(renderer = "iframe")

There is a little bit of right skew in the distribution.

In [ ]:
pipeline.append(("Health Score", __impute_column_with_median, "MEDIAN"))

Let's analyze `Previous Claims`.

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Previous Claims"].value_counts().reset_index(),
    x = "Previous Claims", 
    y = "count", 
    labels={'index': 'Previous Claims', 'Previous Claims': 'Frequency'},
    title="Distribution of Previous Claims",
    template="plotly_dark"
).show(renderer = "iframe")

In [ ]:
pipeline.append(("Previous Claims", __impute_column_with_q1, "Q1"))
pipeline.append(("Previous Claims", __impute_column_with_random_sample_from_distribution, "RANDOM"))

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Vehicle Age"].value_counts().reset_index(),
    x = "Vehicle Age", 
    y = "count", 
    labels={'index': 'Vehicle Age', 'Vehicle Age': 'Frequency'},
    title="Distribution of Vehicle Age",
    template="plotly_dark"
).show(renderer = "iframe")

In [ ]:
pipeline.append(("Vehicle Age", __impute_column_with_median, "MEDIAN"))

In [ ]:
plotly.express.histogram(
    TRAINING_DATAFRAME,
    x="Credit Score",
    nbins=50, 
    labels={'Credit_Score': 'Credit Score', 'count': 'Frequency'},
    title="Histogram of Credit Score",
    template="plotly_dark"
).show(renderer = "iframe")

In [ ]:
pipeline.append(("Credit Score", __impute_column_with_median, "MEDIAN"))

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Insurance Duration"].value_counts().reset_index(),
    x = "Insurance Duration", 
    y = "count", 
    labels={'index': 'Insurance Duration', 'Insurance Duration': 'Frequency'},
    title="Distribution of Insurance Duration",
    template="plotly_dark"
).show(renderer = "iframe")

In [ ]:
pipeline.append(("Insurance Duration", __impute_column_with_median, "MEDIAN"))

In [ ]:
plotly.express.bar(
    TRAINING_DATAFRAME["Customer Feedback"].value_counts().reset_index(),
    x = "Customer Feedback", 
    y = "count", 
    labels={'index': 'Customer Feedback', 'Customer Feedback': 'Frequency'},
    title="Distribution of Customer Feedback",
    template="plotly_dark"
).show(renderer = "iframe")

In [ ]:
pipeline.append(("Customer Feedback", __impute_column_with_value, "W_VALUE"))

Applying the cleaning pipeline.

![](https://helios-i.mashable.com/imagery/articles/03AQw1L3spvJG54ChmQP95o/images-1.fill.size_2000x1125.v1700592699.jpg)
Source: Squid Games | Netflix

In [ ]:
TRAINING_DATAFRAME = run_pipeline(pipeline, TRAINING_DATAFRAME)
TESTING_DATAFRAME  = run_pipeline(pipeline, TESTING_DATAFRAME)

Dropping Column:

In [ ]:
TRAINING_DATAFRAME.columns

In [ ]:
columns_to_drop = [
    'Age_RANDOM',
    'Number of Dependents_RANDOM',
    'Number of Dependents_MEDIAN',
    'Previous Claims_RANDOM',
    'Marital Status_MODE',
    'Marital Status_RANDOM',
    'Occupation_MODE',
    'Occupation_RANDOM'
]

for column in columns_to_drop:
    if column in TRAINING_DATAFRAME.columns:
        TRAINING_DATAFRAME = TRAINING_DATAFRAME.drop(column, axis = "columns")

    if column in TESTING_DATAFRAME.columns:
        TESTING_DATAFRAME = TESTING_DATAFRAME.drop(column, axis = "columns")

### 🤖 3.2 Generate New Features

In [ ]:
def __encode_date(dataframe):
    dataframe['Policy Start Date'] = pandas.to_datetime(dataframe['Policy Start Date'])
    dataframe['Year'] = dataframe['Policy Start Date'].dt.year.astype(str)
    dataframe.drop('Policy Start Date', axis=1, inplace=True)
    return dataframe

TRAINING_DATAFRAME = __encode_date(TRAINING_DATAFRAME)
TESTING_DATAFRAME = __encode_date(TESTING_DATAFRAME)

### 🤖 3.3 One hot encode categorical columns and normalize quantitative data

In [ ]:
training_id = TRAINING_DATAFRAME["id"]
TRAINING_DATAFRAME = TRAINING_DATAFRAME.drop(["id"], axis = "columns")

testing_id = TESTING_DATAFRAME["id"]
TESTING_DATAFRAME = TESTING_DATAFRAME.drop(["id"], axis = "columns")

In [ ]:
for categorical_column in TRAINING_DATAFRAME.describe(include='object').columns:
    encoder = sklearn.preprocessing.OrdinalEncoder()
    TRAINING_DATAFRAME[categorical_column] = TRAINING_DATAFRAME[categorical_column].astype(str)
    TRAINING_DATAFRAME[categorical_column] = encoder.fit_transform(TRAINING_DATAFRAME[[categorical_column]])
    
    TESTING_DATAFRAME[categorical_column] = TESTING_DATAFRAME[categorical_column].astype(str)
    TESTING_DATAFRAME[categorical_column] = encoder.fit_transform(TESTING_DATAFRAME[[categorical_column]])
    

In [ ]:
normalizer = sklearn.preprocessing.StandardScaler()
quantitative_columns = TESTING_DATAFRAME.describe().columns

TRAINING_DATAFRAME[quantitative_columns] = pandas.DataFrame(
    normalizer.fit_transform(TRAINING_DATAFRAME[quantitative_columns])
)
TESTING_DATAFRAME[quantitative_columns] = pandas.DataFrame(
    normalizer.transform(TESTING_DATAFRAME[quantitative_columns])
)

### 3.3 EDA: Todo

### 3.4 🤖 Model Building: Train machine learning models, starting with basic classifiers and then experimenting with more complex models as needed.

In [ ]:
def model_pipeline(model, parameters, training_dataframe, testing_dataframe, testing_id):
    y = training_dataframe["Premium Amount"]
    X = training_dataframe.drop("Premium Amount", axis = "columns")
    
    classifier_ = sklearn.model_selection.GridSearchCV(
        model,
        parameters,
        scoring='neg_mean_squared_error',
    )
    
    classifier_.fit(X, y)
    
    best_params = classifier_.best_params_
    best_score = classifier_.best_score_
    
    print("Best Parameters:", best_params)
    print("Best Score:", best_score)

    return classifier_.best_estimator_, best_params

In [ ]:
cat_features = TRAINING_DATAFRAME.select_dtypes(include=['object', 'bool']).columns.tolist()
lgbm_cat_features = [f"name:{col}" for col in cat_features]

models = [
    {
        "name": "RandomForestRegressor_1",
        "model": sklearn.ensemble.RandomForestRegressor(
            n_estimators=100,
            max_depth=7,
            min_samples_split=10,
            min_samples_leaf=4,
            max_features='sqrt',
            bootstrap=True,
            random_state=42,
            n_jobs=-1 
        ),
        "train": False
    },
    {
        "name": "RandomForestRegressor_2",
        "model": sklearn.ensemble.RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=20,
            min_samples_leaf=5,
            max_features='log2',
            bootstrap=False,
            random_state=42,
            n_jobs=-1 
        ),
        "train": False
    },
    {
        "name": "XGBRegressor_1",
        "model": xgboost.XGBRegressor(
            random_state=42,
            n_estimators=100,
            max_depth=6,
            learning_rate=0.01,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=0.1,
            n_jobs=-1 
        ),
        "train": False
    },
    {
        "name": "XGBRegressor_2",
        "model": xgboost.XGBRegressor(
            random_state=42,
            n_estimators=100,
            max_depth=7,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=1.0,
            gamma=0.2,
            n_jobs=-1 
        ),
        "train": False
    },
    {
        "name": "LGBMRegressor_1",
        "model": lightgbm.LGBMRegressor(
            random_state=42,
            n_estimators=100,
            max_depth=6,
            learning_rate=0.01,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=20,
            n_jobs=-1,
            categorical_feature = lgbm_cat_features
        ),
        "train": True
    },
    {
        "name": "LGBMRegressor_2",
        "model": lightgbm.LGBMRegressor(
            random_state=42,
            n_estimators=100,
            max_depth=8,
            learning_rate=0.05,
            num_leaves=40,
            subsample=0.9,
            colsample_bytree=0.9,
            min_child_samples=30,
            n_jobs=-1,
            categorical_feature = lgbm_cat_features
        ),
        "train": True
    },
    {
        "name": "GradientBoostingRegressor_1",
        "model": sklearn.ensemble.GradientBoostingRegressor(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            min_samples_split=10,
            min_samples_leaf=4,
            random_state=42,
        ),
        "train": False
    },
    {
        "name": "GradientBoostingRegressor_2",
        "model": sklearn.ensemble.GradientBoostingRegressor(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.9,
            min_samples_split=20,
            min_samples_leaf=6,
            random_state=42
        ),
        "train": False
    },
    {
        "name": "CatBoostRegressor_1",
        "model": catboost.CatBoostRegressor(
            depth = 10,
            cat_features = cat_features,
            objective = 'RMSE',
            min_data_in_leaf = 18,
            l2_leaf_reg = 9.044,
            n_estimators=100,
        ),
        "train": True
    },
    {
        "name": "CatBoostRegressor_2",
        "model": catboost.CatBoostRegressor(
            depth = 7,
            cat_features = cat_features,
            objective = 'RMSE',
            min_data_in_leaf = 18,
            l2_leaf_reg = 9.044,
            n_estimators=100,
        ),
        "train": True
    }
]

best_estimators = [(model["name"], model["model"]) for model in models]

In [ ]:
voting_regressor = sklearn.ensemble.StackingRegressor(
    estimators = best_estimators, 
    final_estimator = sklearn.linear_model.Ridge(),
    n_jobs=-1
)

y = TRAINING_DATAFRAME["Premium Amount"]
X = TRAINING_DATAFRAME.drop("Premium Amount", axis="columns")
voting_regressor.fit(X, y)

out = pandas.DataFrame(voting_regressor.predict(TESTING_DATAFRAME))
res = pandas.concat([pandas.DataFrame(testing_id), out], axis=1)
res.columns = ["id", "Premium Amount"]
res[["id", "Premium Amount"]].to_csv("submission.csv", index=False)

### 🧑‍🏫 To be continued 